# Problem Set 6 — Questions (c), (d), (e), (f)

**Scope:** This notebook contains ONLY (c)–(f). The file `Problem_Set6_2026.xlsx` must be in the same folder as this notebook, just as for your teammate's `ps6_a_b.ipynb`.

**Conventions:** monthly returns and factors are in percentage points, not decimals; each momentum strategy is long top 3 and short bottom 3, equally weighted; formation uses only previously observed months. Ties are resolved by original Excel column order (`keep="first"`), matching teammate A. No risk-free subtraction from zero-investment winner-minus-loser returns. FF3/FF4 regressions use matching monthly observations. Ordinary OLS t-statistics (not HAC).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import t as student_t

DATA_PATH = "Problem_Set6_2026.xlsx"

# Match teammate A/B's Excel read conventions.
ind = pd.read_excel(DATA_PATH, sheet_name="30 industries", header=3)
ind = ind.rename(columns={ind.columns[0]: "yyyymm"}).dropna(how="all")
ind["yyyymm"] = ind["yyyymm"].astype(int)
ind.index = pd.PeriodIndex(ind["yyyymm"].astype(str), freq="M", name="date")
R = ind.drop(columns="yyyymm").astype(float).mask(lambda x: x <= -999)
assert R.shape == (1200, 30) and R.notna().all().all()
assert R.index.is_unique and all((b - a).n == 1 for a, b in zip(R.index[:-1], R.index[1:]))

ff = pd.read_excel(DATA_PATH, sheet_name="Fama-French factors", header=2)
ff = ff.rename(columns={ff.columns[0]: "yyyymm"}).dropna(how="all")
ff["yyyymm"] = ff["yyyymm"].astype(int)
ff.index = pd.PeriodIndex(ff["yyyymm"].astype(str), freq="M", name="date")
ff = ff.drop(columns="yyyymm").astype(float).mask(lambda x: x <= -999)
assert ff.index.equals(R.index)
assert {"Mkt-RF", "SMB", "HML", "UMD"}.issubset(ff.columns)
print("Industry returns:", R.shape, R.index.min(), "to", R.index.max())
print("Factors:", ff.columns.tolist(), "| available UMD months:", ff["UMD"].notna().sum())

Industry returns: (1200, 30) 1926-07 to 2026-06
Factors: ['Mkt-RF', 'SMB', 'HML', 'RF', 'UMD'] | available UMD months: 1194


## (c) and (d) — Industry momentum

- **(c) 12–1:** rank on cumulative compounded returns from **t−12 through t−1** (12 months), then take month t winner-minus-loser return.
- **(d) 12–1 skip:** rank on cumulative compounded returns from **t−12 through t−2** (11 months), omitting t−1 entirely, then take month t return.
- A **1–1** monthly series is reconstructed here only as an input to the (e)/(f) regressions. Its selection rule follows teammate A's (a).

In [2]:
def top_minus_bottom(formation_returns: pd.Series, realized_returns: pd.Series) -> float:
    top3 = formation_returns.nlargest(3, keep="first").index
    bottom3 = formation_returns.nsmallest(3, keep="first").index
    assert len(set(top3).intersection(bottom3)) == 0
    return float(realized_returns.loc[top3].mean() - realized_returns.loc[bottom3].mean())

rows = []
for i in range(1, len(R)):
    row = {"date": R.index[i], "1_1": top_minus_bottom(R.iloc[i - 1], R.iloc[i])}
    if i >= 12:
        c_signal = (1 + R.iloc[i - 12:i] / 100).prod(axis=0) - 1  # t-12 ... t-1
        d_signal = (1 + R.iloc[i - 12:i - 1] / 100).prod(axis=0) - 1  # t-12 ... t-2
        row["12_1"] = top_minus_bottom(c_signal, R.iloc[i])
        row["12_1_skip"] = top_minus_bottom(d_signal, R.iloc[i])
    rows.append(row)

mom = pd.DataFrame(rows).set_index("date").reindex(R.index)
assert mom["1_1"].count() == 1199
assert mom["12_1"].count() == mom["12_1_skip"].count() == 1188
assert mom["12_1"].first_valid_index() == pd.Period("1927-07", freq="M")
# Confirm the 1-1 mean agrees with teammate A's notebook (given source data).
assert abs(mom["1_1"].mean() - 0.7380233527939949) < 1e-9

def perf_stats(x):
    x = x.dropna()
    mean, sd = x.mean(), x.std(ddof=1)
    return pd.Series({"start": str(x.index[0]), "end": str(x.index[-1]), "T": len(x),
                      "mean_%_mo": mean, "SD_%_mo": sd,
                      "t_mean": mean / (sd / np.sqrt(len(x))),
                      "Sharpe_monthly": mean / sd})

performance = mom[["12_1", "12_1_skip"]].apply(perf_stats).T
print("(c) and (d):")
display(performance.round(5))
print("Skip minus non-skip mean return (% points/month):",
      round(performance.loc["12_1_skip", "mean_%_mo"] - performance.loc["12_1", "mean_%_mo"], 6))

(c) and (d):


,start,end,T,mean_%_mo,SD_%_mo,t_mean,Sharpe_monthly
12_1,1927-07,2026-06,1188,0.868869,6.326539,4.733653,0.137337
12_1_skip,1927-07,2026-06,1188,0.869296,6.311425,4.747318,0.137734


Skip minus non-skip mean return (% points/month): 0.000426


## (e) and (f) — Fama–French factor regressions

For each long–short industry momentum return series, estimate both:

\[
R^{W-L}_t = lpha_3 + eta_M (R_M-R_f)_t + eta_S SMB_t + eta_H HML_t + \epsilon_t,
\]

\[
R^{W-L}_t = lpha_4 + eta_M (R_M-R_f)_t + eta_S SMB_t + eta_H HML_t + eta_U UMD_t + \epsilon_t.
\]

Use the **same observations** for the nested FF3 and FF4 regressions for each strategy, to isolate the effect of adding UMD. Since the dependent variable is a zero-investment long–short return, do **not** subtract RF again. Standard OLS t-tests are used.

In [3]:
regressors = ["Mkt-RF", "SMB", "HML", "UMD"]
reg_results = []
for strategy in ["1_1", "12_1", "12_1_skip"]:
    joined = pd.concat([mom[strategy].rename("WML"), ff[regressors]], axis=1).dropna()
    y = joined["WML"]
    for model, regcols in [("FF3", regressors[:3]), ("FF4", regressors)]:
        fit = sm.OLS(y, sm.add_constant(joined[regcols], has_constant="add")).fit()
        reg_results.append({"strategy": strategy, "model": model,
                            "start": str(joined.index[0]), "end": str(joined.index[-1]),
                            "T": len(joined), "alpha_%_mo": fit.params["const"],
                            "alpha_t": fit.tvalues["const"], "alpha_p": fit.pvalues["const"],
                            "UMD_beta": fit.params["UMD"] if model == "FF4" else np.nan,
                            "UMD_t": fit.tvalues["UMD"] if model == "FF4" else np.nan,
                            "R2": fit.rsquared})

regression_table = pd.DataFrame(reg_results)
assert regression_table["T"].tolist() == [1194, 1194, 1188, 1188, 1188, 1188]
assert np.all(regression_table.query("model == 'FF4'")["UMD_beta"] > 0)
print("(e) and (f):")
display(regression_table.round(5))

# A few independent benchmarks from the already-audited source workbook.
assert np.isclose(regression_table.iloc[0]["alpha_%_mo"], 0.8000613154787721, atol=1e-9)
assert np.isclose(regression_table.iloc[1]["alpha_%_mo"], 0.6564341661932115, atol=1e-9)
assert np.isclose(regression_table.iloc[3]["alpha_%_mo"], 0.2138572888354315, atol=1e-9)
assert np.isclose(regression_table.iloc[5]["alpha_%_mo"], 0.19310415563910174, atol=1e-9)
print("Checks passed: matches teammate A's 1-1 convention and audited FF3/FF4 alphas.")

(e) and (f):


,strategy,model,start,end,T,alpha_%_mo,alpha_t,alpha_p,UMD_beta,UMD_t,R2
0,1_1,FF3,1927-01,2026-06,1194,0.80006,4.95217,0.00000,NaN,NaN,0.01492
1,1_1,FF4,1927-01,2026-06,1194,0.65643,3.98304,0.00007,0.15083,3.88579,0.02727
2,12_1,FF3,1927-07,2026-06,1188,1.14475,6.46815,0.00000,NaN,NaN,0.09251
3,12_1,FF4,1927-07,2026-06,1188,0.21386,1.58681,0.11282,0.98547,31.06471,0.50021
4,12_1_skip,FF3,1927-07,2026-06,1188,1.11704,6.27739,0.00000,NaN,NaN,0.07820
5,12_1_skip,FF4,1927-07,2026-06,1188,0.19310,1.41045,0.15867,0.97811,30.35110,0.48175


Checks passed: matches teammate A's 1-1 convention and audited FF3/FF4 alphas.


## Answers and AI-integration interpretation

**(c)** The 12–1 strategy uses the previous 12 months' compounded performance to rank industries. The table reports the average return, t-statistic, Sharpe ratio and standard deviation. Its positive mean is statistically significant under a conventional t-test.

**(d)** The skipped-month strategy deliberately excludes t−1; the relevant formation window is **t−12 through t−2**, 11 months under the assignment's explicit definition. In this sample its mean return and Sharpe ratio are almost identical to (c), so avoid over-interpreting a tiny difference.

**(e)** In the FF3 model, all three strategy alphas are positive and significant at 5%, implying that market, size and value factors do not fully price the industry momentum strategies in this sample.

**(f) — prediction before prompting AI:** Since UMD is a traded stock-momentum factor, we expect adding it to reduce industry-momentum alpha, especially for 12–1 signals. **Evaluation:** UMD is positively related to all three strategies, and the two 12–1 variants' FF4 alphas are not significantly different from zero at 5%. However, **1–1 remains significantly positive**. This comparison establishes exposure to a common momentum factor, not why its premium exists. UMD is itself a winner-minus-loser momentum portfolio; using UMD to price industry momentum is partly mechanical. Whether the premium reflects systematic risk or behavioral under-reaction requires separate evidence; these regressions cannot identify the causal mechanism.

**AI integration documentation:** The assignment asks you to first form your own prediction, then prompt an LLM with the FF3/FF4 conceptual question and *critically evaluate its answer* using your regression results. The paragraph above gives the interpretation; before final submission, add a brief truthful statement about what your actual LLM response got right or wrong and what you checked. Do not claim a prompt or check you did not perform.

**Common-sample note:** Each strategy's FF3 and FF4 use the same months when UMD is observed. The 1–1 strategy's earliest five holding months are therefore excluded from this *paired comparison*, although they remain part of teammate A's (a) sample.

## For the (g) teammate (optional handoff)

The variable `mom` holds month-by-month 1–1, 12–1 and 12–1 skip industry momentum. If your teammate needs these series, run the export cell below and share the generated CSV. The CSV is **not required** for this notebook to reproduce (c)–(f).

In [4]:
# Optional — uncomment only when handing data to the (g) teammate:
# mom.to_csv("industry_momentum_monthly.csv", index_label="date")